In [25]:
import pandas as pd 
import matplotlib.pyplot as plt 
import numpy as np

In [26]:
# reading data
default_df = pd.read_csv("trajectory_data/no_connection_time_optimization_default_samples_circle_2_0_obstacle.csv")
connection_time_optimized_df = pd.read_csv("trajectory_data/default_samples_circle_2_0_obstacle.csv")
near_obstacle_df = pd.read_csv("trajectory_data/circle_obstacle_0_1m_away.csv")
fast_speed_far_obstacle_df = pd.read_csv("trajectory_data/fast_speed_far_obstacle.csv")
fast_speed_robot_obstacle_df = pd.read_csv("trajectory_data/fast_speed_robot_obstacle.csv")
start_in_obstacle_df = pd.read_csv("trajectory_data/start_in_circle_obstacle.csv")
df = pd.read_csv("/tmp/tbots/yellow_test/path_summary.csv")
df

,sub_dest_x,sub_dest_y,connection_time,duration,start_x,start_y,end_x,end_y,initial_vel(x),initial_vel(y),...,x20,y20,circle_obst_x,circle_obst_y,circle_obst_rad,rect_obst_x1,rect_obst_y1,rect_obst_x2,rect_obst_y2,total_cost
0,-4.40000,-1.050000,0.0,1.67332,-4.4,-1.05,-4.4,1.05,0,0,...,-4.4,1.05,0,0,0,-4.8,-1,-3.5,1,8.51332
1,-4.30000,-1.050000,0.2,1.90235,-4.4,-1.05,-4.4,1.05,0,0,...,-4.4,1.05,0,0,0,-4.8,-1,-3.5,1,8.54935
2,-4.30865,-1.009330,0.2,1.82403,-4.4,-1.05,-4.4,1.05,0,0,...,-4.4,1.05,0,0,0,-4.8,-1,-3.5,1,8.56762
3,-4.33309,-0.975686,0.2,1.75565,-4.4,-1.05,-4.4,1.05,0,0,...,-4.4,1.05,0,0,0,-4.8,-1,-3.5,1,8.46152
4,-4.36910,-0.954894,0.2,1.74038,-4.4,-1.05,-4.4,1.05,0,0,...,-4.4,1.05,0,0,0,-4.8,-1,-3.5,1,8.58434
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,-2.39261,-3.279430,0.8,3.80873,-4.4,-1.05,-4.4,1.05,0,0,...,-4.4,1.05,0,0,0,-4.8,-1,-3.5,1,3.80873
159,-1.65936,-2.270210,0.2,2.00607,-4.4,-1.05,-4.4,1.05,0,0,...,-4.4,1.05,0,0,0,-4.8,-1,-3.5,1,8.40729
160,-1.65936,-2.270210,0.4,2.44359,-4.4,-1.05,-4.4,1.05,0,0,...,-4.4,1.05,0,0,0,-4.8,-1,-3.5,1,3.54359
161,-1.65936,-2.270210,0.6,2.99485,-4.4,-1.05,-4.4,1.05,0,0,...,-4.4,1.05,0,0,0,-4.8,-1,-3.5,1,3.09485


In [27]:
import matplotlib.pyplot as plt

def plot_trajectories(df, title):

    # Normalize the 'total_cost' column to [0, 1]
    df['normalized_cost'] = (df['total_cost'] - df['total_cost'].min()) / (df['total_cost'].max() - df['total_cost'].min())

    # Plot all other trajectories
    for _, row in df.iterrows():
        x_coords = [row[f'x{i}'] for i in range(21)]
        y_coords = [row[f'y{i}'] for i in range(21)]
        plt.plot(x_coords, y_coords, color='black', alpha=0.1 + 0.4 * (1-row['normalized_cost']), zorder=1)

    # Find the row with the lowest total cost
    min_cost_row = df.loc[df['total_cost'].idxmin()]

    # Extract the x and y coordinates of the trajectory
    x_coords_min = [min_cost_row[f'x{i}'] for i in range(21)]
    y_coords_min = [min_cost_row[f'y{i}'] for i in range(21)]

    # Plot the trajectory with the lowest cost
    plt.plot(x_coords_min, y_coords_min, label='Lowest Cost Trajectory', color='orange', linewidth=3)

    # Plot the circle obstacle
    if min_cost_row['circle_obst_rad'] != 0:
        circle = plt.Circle((min_cost_row['circle_obst_x'], min_cost_row['circle_obst_y']), min_cost_row['circle_obst_rad'], color='r', label='Obstacle', fill=False, linewidth=2)
        plt.gca().add_patch(circle)

    if 'rect_obst_x1' in min_cost_row:
        width = abs(min_cost_row['rect_obst_x1'] - min_cost_row['rect_obst_x2'])
        height = abs(min_cost_row['rect_obst_y1'] - min_cost_row['rect_obst_y2'])
        rectangle = plt.Rectangle((min_cost_row['rect_obst_x1'], min_cost_row['rect_obst_y1']), width, height, label='Obstacle', color='r', fill=False, linewidth=2)
        plt.gca().add_patch(rectangle)

    # Plot start and end positions
    start = plt.Circle((min_cost_row['start_x'], min_cost_row['start_y']), 0.09, color='g', label='Start', fill=True)
    end = plt.Circle((min_cost_row['end_x'], min_cost_row['end_y']), 0.09, color='purple', label='End', fill=True)
    plt.gca().add_patch(start)
    plt.gca().add_patch(end)

    # Draw a vector to represent the initial velocity
    if min_cost_row['initial_vel(x)'] != 0 or min_cost_row['initial_vel(y)'] != 0:
        plt.arrow(min_cost_row['start_x'], min_cost_row['start_y'], min_cost_row['initial_vel(x)'], min_cost_row['initial_vel(y)'], color='blue', width=0.04, label='Initial velocity')    

    plt.xlabel('x')
    plt.ylabel('y')
    # Add description text outside of the plot
    plt.figtext(0.5, -0.04, 'Black lines represent the sub-optimal trajectories (Darker is better)', ha='center')
    plt.figtext(0.5, -0.08, f'{len(df)-1} trajectories sampled', ha='center')
    plt.title(title)
    plt.legend()

    # Set equal aspect ratio
    plt.axis('equal')
    plt.show()

plot_trajectories(df, title="Sampled Trajectories - without connection_time optimization")
# plot_trajectories(default_df, title='Sampled Trajectories - without connection_time optimization')
# plot_trajectories(connection_time_optimized_df, title='Sampled Trajectories - with connection_time optimization')
# plot_trajectories(near_obstacle_df, title='Sampled Trajectories - with connection_time optimization')
# plot_trajectories(fast_speed_far_obstacle_df, title='Sampled Trajectories - with connection_time optimization')
# plot_trajectories(fast_speed_robot_obstacle_df, title='Sampled Trajectories - with connection_time optimization')
# plot_trajectories(start_in_obstacle_df, title='Sampled Trajectories - with connection_time optimization')

